# Import Required Libraries
Import the necessary libraries, including TensorFlow, Keras, and other dependencies.

In [1]:
# Importieren der benötigten Bibliotheken
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import io
import base64
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import pymongo
import os
from tensorflow.keras.utils import Sequence, to_categorical
import gc
from tqdm import tqdm
import tensorflow as tf

# Funktion zur Datenbankverbindung und Datenladung
def load_data():
    mongo_uri = os.getenv('MONGO_URI', 'mongodb://localhost:27017/fingerprintDB')
    client = pymongo.MongoClient(mongo_uri)
    db = client.get_default_database()
    
    # Daten laden
    canvassamples = list(db['canvassamples'].find())
    fingerprints = list(db['fingerprints'].find())
    
    # In DataFrames konvertieren
    canvassamples_df = pd.DataFrame(canvassamples)
    fingerprints_df = pd.DataFrame(fingerprints)
    
    # Merge der DataFrames
    merged_df = pd.merge(canvassamples_df, fingerprints_df, 
                        left_on='fingerprintId', 
                        right_on='_id', 
                        suffixes=('_sample', '_fingerprint'))
    
    return merged_df

# Daten laden
merged_df = load_data()
print(f"Gesamtdatensatz enthält {len(merged_df)} Einträge.")

Gesamtdatensatz enthält 2000000 Einträge.


In [2]:
# Benutzer-IDs extrahieren
user_ids = merged_df['username'].unique()

# DataFrames für jeden Benutzer erstellen
user_dfs = {user_id: merged_df[merged_df['username'] == user_id] for user_id in user_ids}

# Beispielbenutzer auswählen (z.B. 'benutzername_1')
example_user_id = 'benutzername_1'

# Positive Beispiele (Beispielbenutzer)
user_df = user_dfs[example_user_id].sample(n=15000, random_state=42)

# Negative Beispiele (andere Benutzer)
negative_df = pd.concat([user_dfs[user_id] for user_id in user_ids if user_id != example_user_id])
negative_df = negative_df.sample(n=10000, random_state=42)

# Aufteilung in Trainings-, Validierungs- und Testdaten
train_df, test_df = train_test_split(pd.concat([user_df, negative_df]), test_size=0.1, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2222, random_state=42)  # 0.2222 * 0.9 ≈ 0.2

In [3]:
# Funktion zur Verarbeitung der Bilder in RGB
def process_image_rgb(base64_str, target_size=(224, 224)):
    try:
        image_data = base64.b64decode(base64_str)
        image = Image.open(io.BytesIO(image_data)).convert('RGB')  # Konvertiere Bild zu RGB
        image = image.resize(target_size)
        image_array = np.array(image) / 255.0  # Normalisierung auf Werte zwischen 0 und 1
        return image_array
    except Exception as e:
        print(f"Fehler bei der Bildverarbeitung: {e}")
        return None

def extract_images_from_df(df):
    images = []
    labels = []
    for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Lade Bilder"):
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            images.append(image)
            labels.append(1 if row['username'] == example_user_id else 0)
    return np.array(images), np.array(labels)

# Bilddaten verarbeiten
X_train, y_train = extract_images_from_df(train_df)
X_val, y_val = extract_images_from_df(val_df)
X_test, y_test = extract_images_from_df(test_df)

# Speicher freigeben
gc.collect()

Lade Bilder: 100%|██████████| 2500/2500 [00:01<00:00, 1432.63it/s]


0

In [ ]:
# Einfaches CNN-Modell definieren
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

# Modell kompilieren
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Modellübersicht anzeigen
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 111, 111, 32)      0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 54, 54, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 52, 52, 128)       73856     
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 26, 26, 128)       0

2024-11-22 15:52:00.444334: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-11-22 15:52:00.444499: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-11-22 15:52:00.459229: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

: 

In [5]:
# Early Stopping Callback definieren
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Modell trainieren
history = model.fit(
    X_train, y_train,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)

: 

: 

In [ ]:
# Modell evaluieren
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Vorhersagen auf Testdaten
y_pred = model.predict(X_test)
y_pred_classes = (y_pred > 0.5).astype("int32").flatten()
y_true = y_test

# Klassifikationsbericht
print(classification_report(y_true, y_pred_classes))

# Konfusionsmatrix
conf_matrix = confusion_matrix(y_true, y_pred_classes)
sns.heatmap(conf_matrix, annot=True, fmt='d')
plt.title('Konfusionsmatrix')
plt.xlabel('Vorhergesagte Klasse')
plt.ylabel('Wahre Klasse')
plt.show()

# ROC-Kurve
fpr, tpr, _ = roc_curve(y_true, y_pred)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
import random

# Funktion zur Auswahl zufälliger Samples von verschiedenen Benutzern
def get_random_samples(user_dfs, example_user_id, num_users=5, num_samples_per_user=10):
    selected_users = random.sample([user_id for user_id in user_dfs.keys() if user_id != example_user_id], num_users)
    samples = []
    labels = []
    user_labels = []
    
    for user_id in selected_users:
        user_samples = user_dfs[user_id].sample(n=num_samples_per_user, random_state=42)
        for _, row in user_samples.iterrows():
            image = process_image_rgb(row['sampleData'])
            if image is not None:
                samples.append(image)
                labels.append(0)  # Negative Beispiel
                user_labels.append(user_id)
    
    # Füge positive Beispiele hinzu
    user_samples = user_dfs[example_user_id].sample(n=num_samples_per_user, random_state=42)
    for _, row in user_samples.iterrows():
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            samples.append(image)
            labels.append(1)  # Positive Beispiel
            user_labels.append(example_user_id)
    
    return np.array(samples), np.array(labels), user_labels

# Funktion zur Analyse des Modells
def analyze_model(model, user_dfs, example_user_id, iterations=5, num_users=5, num_samples_per_user=10):
    all_results = []
    
    for i in range(iterations):
        X_samples, y_samples, user_labels = get_random_samples(user_dfs, example_user_id, num_users, num_samples_per_user)
        y_pred = model.predict(X_samples)
        y_pred_classes = (y_pred > 0.5).astype("int32").flatten()
        
        # Klassifikationsbericht
        print(f"Iteration {i+1}")
        print(classification_report(y_samples, y_pred_classes))
        
        # Konfusionsmatrix
        conf_matrix = confusion_matrix(y_samples, y_pred_classes)
        sns.heatmap(conf_matrix, annot=True, fmt='d')
        plt.title(f'Konfusionsmatrix - Iteration {i+1}')
        plt.xlabel('Vorhergesagte Klasse')
        plt.ylabel('Wahre Klasse')
        plt.show()
        
        # ROC-Kurve
        fpr, tpr, _ = roc_curve(y_samples, y_pred)
        roc_auc = auc(fpr, tpr)
        plt.figure()
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve - Iteration {i+1}')
        plt.legend(loc="lower right")
        plt.show()
        
        all_results.append((y_samples, y_pred_classes, user_labels))
    
    return all_results

# Analyse durchführen
results = analyze_model(model, user_dfs, example_user_id)

# Funktion zur grafischen Darstellung der Benutzerklassifikationen
def plot_user_classifications_all_iterations(results, example_user_id):
    combined_df = pd.DataFrame()
    
    for i, (y_samples, y_pred_classes, user_labels) in enumerate(results):
        df = pd.DataFrame({'User': user_labels, 'True Label': y_samples, 'Predicted Label': y_pred_classes})
        df['Correct'] = df['True Label'] == df['Predicted Label']
        df['Iteration'] = i + 1
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    
    plt.figure(figsize=(14, 8))
    sns.countplot(data=combined_df, x='User', hue='Correct', palette={True: 'green', False: 'red'}, dodge=True)
    plt.title('User Classification Results Across All Iterations')
    plt.xlabel('User')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.legend(title='Correct Prediction')
    plt.show()

# Grafische Darstellung der Benutzerklassifikationen
plot_user_classifications_all_iterations(results, example_user_id)

In [ ]:
import random

# Funktion zur Auswahl zufälliger Samples von verschiedenen Benutzern
def get_random_samples(user_dfs, example_user_id, num_users=5, num_samples_per_user=10):
    selected_users = random.sample([user_id for user_id in user_dfs.keys() if user_id != example_user_id], num_users)
    samples = []
    labels = []
    user_labels = []
    
    for user_id in selected_users:
        user_samples = user_dfs[user_id].sample(n=num_samples_per_user, random_state=42)
        for _, row in user_samples.iterrows():
            image = process_image_rgb(row['sampleData'])
            if image is not None:
                samples.append(image)
                labels.append(0)  # Negative Beispiel
                user_labels.append(user_id)
    
    # Füge positive Beispiele hinzu
    user_samples = user_dfs[example_user_id].sample(n=num_samples_per_user, random_state=42)
    for _, row in user_samples.iterrows():
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            samples.append(image)
            labels.append(1)  # Positive Beispiel
            user_labels.append(example_user_id)
    
    return np.array(samples), np.array(labels), user_labels

# Funktion zur Analyse des Modells
def analyze_model(model, user_dfs, example_user_id, iterations=5, num_users=5, num_samples_per_user=10):
    all_results = []
    
    for i in range(iterations):
        X_samples, y_samples, user_labels = get_random_samples(user_dfs, example_user_id, num_users, num_samples_per_user)
        y_pred = model.predict(X_samples)
        y_pred_classes = (y_pred > 0.5).astype("int32").flatten()
        
        # Klassifikationsbericht
        print(f"Iteration {i+1}")
        print(classification_report(y_samples, y_pred_classes))
        
        # Konfusionsmatrix
        conf_matrix = confusion_matrix(y_samples, y_pred_classes)
        sns.heatmap(conf_matrix, annot=True, fmt='d')
        plt.title(f'Konfusionsmatrix - Iteration {i+1}')
        plt.xlabel('Vorhergesagte Klasse')
        plt.ylabel('Wahre Klasse')
        plt.show()
        
        # ROC-Kurve
        fpr, tpr, _ = roc_curve(y_samples, y_pred)
        roc_auc = auc(fpr, tpr)
        plt.figure()
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve - Iteration {i+1}')
        plt.legend(loc="lower right")
        plt.show()
        
        all_results.append((y_samples, y_pred_classes, user_labels))
    
    return all_results

# Analyse durchführen
results = analyze_model(model, user_dfs, example_user_id)

# Funktion zur grafischen Darstellung der Benutzerklassifikationen
def plot_user_classifications_all_iterations(results, example_user_id):
    combined_df = pd.DataFrame()
    
    for i, (y_samples, y_pred_classes, user_labels) in enumerate(results):
        df = pd.DataFrame({'User': user_labels, 'True Label': y_samples, 'Predicted Label': y_pred_classes})
        df['Correct'] = df['True Label'] == df['Predicted Label']
        df['Iteration'] = i + 1
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    
    plt.figure(figsize=(14, 8))
    sns.countplot(data=combined_df, x='User', hue='Correct', palette={True: 'green', False: 'red'}, dodge=True)
    plt.title('User Classification Results Across All Iterations')
    plt.xlabel('User')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.legend(title='Correct Prediction')
    plt.show()

# Grafische Darstellung der Benutzerklassifikationen
plot_user_classifications_all_iterations(results, example_user_id)

# Modellarchitekturen anzeigen
for model_name, model in models.items():
    print(f"Architektur von {model_name}:")
    model.summary()

# Statistiken anzeigen
def display_statistics(evaluation_results):
    for model_name, result in evaluation_results.items():
        print(f"Statistiken für {model_name}:")
        print(result['classification_report'])
        cm = result['confusion_matrix']
        tn, fp, fn, tp = cm.ravel()
        print(f"True Positive: {tp}, False Positive: {fp}, False Negative: {fn}, True Negative: {tn}")
        print("\n")

display_statistics(evaluation_results)

# Funktion zur Berechnung und Anzeige der Score-Bewertung
def calculate_scores(evaluation_results):
    scores = {}
    for model_name, result in evaluation_results.items():
        cm = result['confusion_matrix']
        tn, fp, fn, tp = cm.ravel()
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        precision = tp / (tp + fp) if (tp + fp) != 0 else 0
        recall = tp / (tp + fn) if (tp + fn) != 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
        scores[model_name] = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1_score
        }
    return scores

# Score-Bewertung berechnen und anzeigen
scores = calculate_scores(evaluation_results)
for model_name, score in scores.items():
    print(f"Score-Bewertung für {model_name}:")
    for metric, value in score.items():
        print(f"{metric}: {value:.4f}")
    print("\n")

In [ ]:
import random

# Funktion zur Auswahl zufälliger Samples von verschiedenen Benutzern
def get_random_samples(user_dfs, example_user_id, num_users=5, num_samples_per_user=10):
    selected_users = random.sample([user_id for user_id in user_dfs.keys() if user_id != example_user_id], num_users)
    samples = []
    labels = []
    user_labels = []
    
    for user_id in selected_users:
        user_samples = user_dfs[user_id].sample(n=num_samples_per_user, random_state=42)
        for _, row in user_samples.iterrows():
            image = process_image_rgb(row['sampleData'])
            if image is not None:
                samples.append(image)
                labels.append(0)  # Negative Beispiel
                user_labels.append(user_id)
    
    # Füge positive Beispiele hinzu
    user_samples = user_dfs[example_user_id].sample(n=num_samples_per_user, random_state=42)
    for _, row in user_samples.iterrows():
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            samples.append(image)
            labels.append(1)  # Positive Beispiel
            user_labels.append(example_user_id)
    
    return np.array(samples), np.array(labels), user_labels

# Funktion zur Analyse des Modells
def analyze_model(model, user_dfs, example_user_id, iterations=5, num_users=5, num_samples_per_user=10):
    all_results = []
    
    for i in range(iterations):
        X_samples, y_samples, user_labels = get_random_samples(user_dfs, example_user_id, num_users, num_samples_per_user)
        y_pred = model.predict(X_samples)
        y_pred_classes = (y_pred > 0.5).astype("int32").flatten()
        
        # Klassifikationsbericht
        print(f"Iteration {i+1}")
        print(classification_report(y_samples, y_pred_classes))
        
        # Konfusionsmatrix
        conf_matrix = confusion_matrix(y_samples, y_pred_classes)
        sns.heatmap(conf_matrix, annot=True, fmt='d')
        plt.title(f'Konfusionsmatrix - Iteration {i+1}')
        plt.xlabel('Vorhergesagte Klasse')
        plt.ylabel('Wahre Klasse')
        plt.show()
        
        # ROC-Kurve
        fpr, tpr, _ = roc_curve(y_samples, y_pred)
        roc_auc = auc(fpr, tpr)
        plt.figure()
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve - Iteration {i+1}')
        plt.legend(loc="lower right")
        plt.show()
        
        all_results.append((y_samples, y_pred_classes, user_labels))
    
    return all_results

# Analyse durchführen
results = analyze_model(model, user_dfs, example_user_id)

# Funktion zur grafischen Darstellung der Benutzerklassifikationen
def plot_user_classifications_all_iterations(results, example_user_id):
    combined_df = pd.DataFrame()
    
    for i, (y_samples, y_pred_classes, user_labels) in enumerate(results):
        df = pd.DataFrame({'User': user_labels, 'True Label': y_samples, 'Predicted Label': y_pred_classes})
        df['Correct'] = df['True Label'] == df['Predicted Label']
        df['Iteration'] = i + 1
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    
    plt.figure(figsize=(14, 8))
    sns.countplot(data=combined_df, x='User', hue='Correct', palette={True: 'green', False: 'red'}, dodge=True)
    plt.title('User Classification Results Across All Iterations')
    plt.xlabel('User')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.legend(title='Correct Prediction')
    plt.show()

# Grafische Darstellung der Benutzerklassifikationen
plot_user_classifications_all_iterations(results, example_user_id)

# Modellarchitekturen anzeigen
for model_name, model in models.items():
    print(f"Architektur von {model_name}:")
    model.summary()

# Statistiken anzeigen
def display_statistics(evaluation_results):
    for model_name, result in evaluation_results.items():
        print(f"Statistiken für {model_name}:")
        print(result['classification_report'])
        cm = result['confusion_matrix']
        tn, fp, fn, tp = cm.ravel()
        print(f"True Positive: {tp}, False Positive: {fp}, False Negative: {fn}, True Negative: {tn}")
        print("\n")

display_statistics(evaluation_results)

# Funktion zur Berechnung und Anzeige der Score-Bewertung
def calculate_scores(evaluation_results):
    scores = {}
    for model_name, result in evaluation_results.items():
        cm = result['confusion_matrix']
        tn, fp, fn, tp = cm.ravel()
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        precision = tp / (tp + fp) if (tp + fp) != 0 else 0
        recall = tp / (tp + fn) if (tp + fn) != 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
        scores[model_name] = {
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1_score
        }
    return scores

# Score-Bewertung berechnen und anzeigen
scores = calculate_scores(evaluation_results)
for model_name, score in scores.items():
    print(f"Score-Bewertung für {model_name}:")
    for metric, value in score.items():
        print(f"{metric}: {value:.4f}")
    print("\n")

In [ ]:
# Ergebnisse in Tortendiagrammen plotten
def plot_results_pie_charts(results):
    for model_name, result in results.items():
        cm = result['confusion_matrix']
        tn, fp, fn, tp = cm.ravel()
        sizes = [tp, fp, fn, tn]
        labels = ['True Positive', 'False Positive', 'False Negative', 'True Negative']
        plt.figure(figsize=(8, 8))
        plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140)
        plt.title(f'{model_name} Results')
        plt.axis('equal')
        plt.show()

plot_results_pie_charts(evaluation_results)